In [10]:
import pandas as pd
import re
import joblib
import sys
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

sys.path.append('../src')
from text_features import url_tokenizer

df = pd.read_csv('../data/malicious_phish.csv')
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)

with open('../data/whitelist_domains.txt', 'r') as f:
    trusted_domains = [line.strip() for line in f if line.strip()]

print(f"Dataset original : {df.shape[0]} lignes")
print(f"Domaines de confiance disponibles : {len(trusted_domains)}")

augmented_urls = []
for d in trusted_domains:
    augmented_urls.append(d)
    augmented_urls.append(f"www.{d}")
    augmented_urls.append(f"https://{d}")

# Répété 40x pour donner un vrai poids statistique face aux 651k
# lignes existantes (1er essai à 0.24% du dataset, noyé, quasi sans effet)
REPEAT_FACTOR = 40
df_augmented = pd.DataFrame({
    'url': augmented_urls * REPEAT_FACTOR,
    'type': 'benign',
    'label': 0
})

df_v4 = pd.concat([df, df_augmented], ignore_index=True)
print(f"Dataset augmenté : {df_v4.shape[0]} lignes (+{df_augmented.shape[0]} lignes, {df_augmented.shape[0]/df_v4.shape[0]*100:.1f}% du total)")

Dataset original : 651191 lignes
Domaines de confiance disponibles : 530
Dataset augmenté : 714791 lignes (+63600 lignes, 8.9% du total)


In [2]:
IP_PATTERN = re.compile(
    r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$'
)

SHORTENER_PATTERN = re.compile(
    r'bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|'
    r'yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|'
    r'short\.to|budurl\.com|ping\.fm|post\.ly|just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|'
    r'doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|lnkd\.in|'
    r'db\.tt|qr\.ae|adf\.ly|bitly\.com|cur\.lv|ity\.im|q\.gs|po\.st|bc\.vc|twitthis\.com|'
    r'u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|prettylinkpro\.com|scrnch\.me|'
    r'filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|link\.zip\.net'
)


def is_ip_address(domain):
    host = domain.split(':')[0]
    return int(bool(IP_PATTERN.match(host)))


def has_numeric_label(domain):
    host = domain.split(':')[0]
    return int(any(part.isdigit() for part in host.split('.')))


def has_shortener(url):
    return int(bool(SHORTENER_PATTERN.search(url.lower())))


def extract_features_v3(url):
    try:
        if not url.startswith('http'):
            url = 'http://' + url
        parsed = urlparse(url)
        domain = parsed.netloc
        path = parsed.path
    except Exception:
        domain = ''
        path = ''

    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.free.fr', '.xyz', '.top']
    suspicious_words = ['login', 'secure', 'update', 'verify', 'account',
                        'banking', 'paypal', 'amazon', 'apple', 'microsoft']

    return {
        'length': len(url),
        'num_dots': url.count('.'),
        'num_hyphens': url.count('-'),
        'num_slashes': url.count('/'),
        'num_digits': sum(c.isdigit() for c in url),
        'num_at': url.count('@'),
        'num_percent': url.count('%'),
        'num_question': url.count('?'),
        'num_equal': url.count('='),
        'num_underscore': url.count('_'),
        'has_https': int(url.startswith('https')),
        'has_ip': is_ip_address(domain),
        'has_numeric_label': has_numeric_label(domain),
        'domain_length': len(domain),
        'path_length': len(path),
        'has_suspicious_tld': int(any(url.endswith(tld) or tld + '/' in url for tld in suspicious_tlds)),
        'has_suspicious_word': int(any(word in domain.lower() for word in suspicious_words)),
        'num_subdomains': len(domain.split('.')) - 2 if len(domain.split('.')) > 2 else 0,
        'has_at_in_url': int('@' in url),
        'has_shortener': has_shortener(url),
    }


print("Fonctions définies")

Fonctions définies


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df_v4['url'], df_v4['label'], test_size=0.2, random_state=42, stratify=df_v4['label']
)

X_train_feat = pd.DataFrame([extract_features_v3(u) for u in X_train])
X_test_feat = pd.DataFrame([extract_features_v3(u) for u in X_test])

print("Entraînement v4...")
rf_model_v4 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_v4.fit(X_train_feat, y_train)

y_pred = rf_model_v4.predict(X_test_feat)
print(f"Accuracy v4 : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

Entraînement v4...
Accuracy v4 : 91.64%
              precision    recall  f1-score   support

      Benign       0.93      0.95      0.94     98341
   Malicious       0.88      0.85      0.86     44618

    accuracy                           0.92    142959
   macro avg       0.91      0.90      0.90    142959
weighted avg       0.92      0.92      0.92    142959



In [12]:
test_cases = ['google.com', 'github.com', 'stackoverflow.com', 'microsoft.com',
              'wikipedia.org', 'amazon.com', 'linkedin.com']

for url in test_cases:
    feat = pd.DataFrame([extract_features_v3(url)])
    pred = rf_model_v4.predict(feat)[0]
    proba = rf_model_v4.predict_proba(feat)[0]
    verdict = 'MALICIOUS' if pred == 1 else 'BENIGN'
    print(f"{url:25s} -> {verdict:10s} (confiance: {max(proba)*100:.1f}%)")

google.com                -> BENIGN     (confiance: 87.1%)
github.com                -> BENIGN     (confiance: 87.1%)
stackoverflow.com         -> MALICIOUS  (confiance: 70.5%)
microsoft.com             -> BENIGN     (confiance: 100.0%)
wikipedia.org             -> BENIGN     (confiance: 57.4%)
amazon.com                -> BENIGN     (confiance: 100.0%)
linkedin.com              -> BENIGN     (confiance: 66.8%)


In [13]:
joblib.dump(rf_model_v4, '../models/rf_phishing_model_v4.pkl')
print("Modèle v4 sauvegardé")

Modèle v4 sauvegardé
